# RDKit: cheminformatics in your browser

[RDKit](https://www.rdkit.org) is the most widely used open-source cheminformatics toolkit. This notebook parses molecules from SMILES strings, draws them inline, computes descriptors and fingerprints, and runs a substructure search, all in the browser.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Draw, rdFingerprintGenerator
from rdkit.Chem.Draw import IPythonConsole

IPythonConsole.ipython_useSVG = True  # crisp SVG drawings in the notebook

## A molecule displays as a picture

Just put a `Mol` at the end of a cell.

In [ ]:
caffeine = Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C(=O)N2C)C")
caffeine

## A small compound collection

In [ ]:
smiles = {
    "aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "paracetamol": "CC(=O)NC1=CC=C(C=C1)O",
    "ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "theobromine": "CN1C=NC2=C1C(=O)NC(=O)N2C",
    "nicotine": "CN1CCCC1C2=CN=CC=C2",
    "dopamine": "NCCC1=CC(O)=C(O)C=C1",
    "serotonin": "NCCC1=CNC2=C1C=C(O)C=C2",
}
mols = {name: Chem.MolFromSmiles(s) for name, s in smiles.items()}

Draw.MolsToGridImage(list(mols.values()), legends=list(mols), molsPerRow=4, subImgSize=(220, 180))

## Descriptors

Molecular weight, lipophilicity (logP), polar surface area and hydrogen-bond donors/acceptors, as used in Lipinski's rule of five.

In [ ]:
table = pd.DataFrame(
    {
        name: {
            "MW": Descriptors.MolWt(m),
            "logP": Descriptors.MolLogP(m),
            "TPSA": Descriptors.TPSA(m),
            "H donors": Descriptors.NumHDonors(m),
            "H acceptors": Descriptors.NumHAcceptors(m),
        }
        for name, m in mols.items()
    }
).T.round(2)
table

## Similarity

Morgan fingerprints and Tanimoto similarity between all pairs. Caffeine and theobromine (found in chocolate) are close relatives.

In [ ]:
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [generator.GetFingerprint(m) for m in mols.values()]
similarity = pd.DataFrame(
    [DataStructs.BulkTanimotoSimilarity(fp, fps) for fp in fps],
    index=list(mols),
    columns=list(mols),
)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(similarity, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(mols)), list(mols), rotation=45, ha="right")
ax.set_yticks(range(len(mols)), list(mols))
fig.colorbar(im, label="Tanimoto similarity")
ax.set_title("Morgan fingerprint similarity")
fig.tight_layout()

## Substructure search

Which molecules contain a benzene ring with a hydroxyl group (a phenol)? The matching atoms are highlighted.

In [ ]:
phenol = Chem.MolFromSmarts("c[OX2H]")
hits = {name: m for name, m in mols.items() if m.HasSubstructMatch(phenol)}
highlights = [sum(m.GetSubstructMatches(phenol), ()) for m in hits.values()]

Draw.MolsToGridImage(
    list(hits.values()),
    legends=list(hits),
    highlightAtomLists=highlights,
    molsPerRow=3,
    subImgSize=(240, 200),
)

## 3D coordinates

Embed a 3D conformer, optimize it with the MMFF force field and look at the interatomic distances.

In [ ]:
mol3d = Chem.AddHs(mols["ibuprofen"])
AllChem.EmbedMolecule(mol3d, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol3d)

conf = mol3d.GetConformer()
heavy = [a.GetIdx() for a in mol3d.GetAtoms() if a.GetAtomicNum() > 1]
pos = conf.GetPositions()[heavy]

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
colors = {6: "dimgray", 8: "red", 7: "blue"}
ax.scatter(*pos.T, s=120, c=[colors.get(mol3d.GetAtomWithIdx(i).GetAtomicNum(), "green") for i in heavy])
for bond in mol3d.GetBonds():
    i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    if i in heavy and j in heavy:
        p = conf.GetPositions()[[i, j]]
        ax.plot(*p.T, color="black")
ax.set_title("ibuprofen, MMFF-optimized conformer")
ax.set_axis_off()

More examples: the [RDKit Getting Started guide](https://www.rdkit.org/docs/GettingStartedInPython.html).